In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("lakehouse").getOrCreate()



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/17 15:32:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [9]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Grab your existing active notebook Spark Session
spark = SparkSession.builder.getOrCreate()

# 2. Path to two files directly from your bronze bucket
test_paths = [
    "s3a://lakehouse/bronze/ohlcv/1min/AAPL_raw_1min.parquet",
    "s3a://lakehouse/bronze/ohlcv/1min/ABNB_raw_1min.parquet"
]

print("--- Reading raw parquet test files ---")
# Explicitly selecting the hidden column '_metadata' alongside everything else
df_raw = spark.read.parquet(*test_paths).select("*", "_metadata")

# 3. Apply the bulletproof hidden metadata extraction regex
df_with_ticker = df_raw.withColumn(
    "ticker", 
    F.regexp_extract(F.col("_metadata.file_name"), r"([^/]+)_raw", 1)
)

print("\n--- [TEST 1] Checking Ticker Extraction Before Shuffle ---")
# Changed "date" to "datetime" to match your schema
df_with_ticker.select("datetime", "Open", "Close", "ticker").show(5)

# 4. Stress Test: Trigger a heavy window partition
# Changed "date" to "datetime" here as well
window_spec = Window.partitionBy("ticker", "datetime").orderBy(F.lit(1))
df_shuffled = df_with_ticker.withColumn("_row_num", F.row_number().over(window_spec)) \
                            .filter(F.col("_row_num") == 1) \
                            .drop("_row_num")

print("\n--- [TEST 2] Checking Ticker Columns After Shuffle Boundary ---")
df_shuffled.select("datetime", "Open", "Close", "ticker").show(5)

--- Reading raw parquet test files ---

--- [TEST 1] Checking Ticker Extraction Before Shuffle ---
+-------------------+----+-----+------+
|           datetime|Open|Close|ticker|
+-------------------+----+-----+------+
|1041240600000000000|0.23| 0.23|  AAPL|
|1041240660000000000|0.23| 0.23|  AAPL|
|1041240720000000000|0.23| 0.23|  AAPL|
|1041240780000000000|0.23| 0.23|  AAPL|
|1041240840000000000|0.23| 0.23|  AAPL|
+-------------------+----+-----+------+
only showing top 5 rows


--- [TEST 2] Checking Ticker Columns After Shuffle Boundary ---


[Stage 11:==================================================>     (10 + 1) / 11]

+-------------------+----+-----+------+
|           datetime|Open|Close|ticker|
+-------------------+----+-----+------+
|1041241140000000000|0.23| 0.23|  AAPL|
|1041241500000000000|0.23| 0.23|  AAPL|
|1041241740000000000|0.23| 0.23|  AAPL|
|1041242280000000000|0.23| 0.23|  AAPL|
|1041242340000000000|0.23| 0.23|  AAPL|
+-------------------+----+-----+------+
only showing top 5 rows



In [6]:
df  =spark.read.parquet("s3a://lakehouse/bronze/ohlcv/1min/date=2026-06-13/MPWR.parquet")

In [7]:
df.show()


+-------------------+----+----+----+-----+------+---------+------+
|           datetime|Open|High| Low|Close|Volume|   source|ticker|
+-------------------+----+----+----+-----+------+---------+------+
|1100862000000000000| 8.0| 8.0|7.66| 7.79|582126|pitrading|  MPWR|
|1100862060000000000|7.74|7.99|7.52| 7.55|274556|pitrading|  MPWR|
|1100862120000000000|7.58| 7.6|7.51| 7.51|251040|pitrading|  MPWR|
|1100862180000000000|7.51|7.64|7.51| 7.64|159556|pitrading|  MPWR|
|1100862240000000000|7.61|7.73|7.57| 7.65|100792|pitrading|  MPWR|
|1100862300000000000|7.69|7.69|7.55| 7.55|119102|pitrading|  MPWR|
|1100862360000000000|7.55|7.62|7.51| 7.52| 59560|pitrading|  MPWR|
|1100862420000000000|7.52|7.58|7.52| 7.57| 42860|pitrading|  MPWR|
|1100862480000000000|7.55|7.66|7.55| 7.59| 31212|pitrading|  MPWR|
|1100862540000000000|7.62|7.62|7.59| 7.59| 23540|pitrading|  MPWR|
|1100862600000000000|7.59|7.77|7.59| 7.73| 65925|pitrading|  MPWR|
|1100862660000000000|7.71|7.75|7.68| 7.71| 47367|pitrading|  M

In [20]:
df  =spark.read.parquet("s3a://lakehouse/bronze/ohlcv/1min/date=2026-06-04/A.parquet")
from pyspark.sql import functions as F

df = df.withColumn(
    "timestamp",
    F.to_timestamp(
        F.from_unixtime(
            F.col("datetime") / F.lit(1_000_000_000)
        )
    )
)

df.select("datetime", "timestamp").show(5, truncate=False)

+-------------------+-------------------+
|datetime           |timestamp          |
+-------------------+-------------------+
|1041240660000000000|2002-12-30 09:31:00|
|1041240720000000000|2002-12-30 09:32:00|
|1041240780000000000|2002-12-30 09:33:00|
|1041240840000000000|2002-12-30 09:34:00|
|1041240900000000000|2002-12-30 09:35:00|
+-------------------+-------------------+
only showing top 5 rows



In [36]:
df = spark.read.parquet("s3a://lakehouse/bronze/ohlcv/1min")
df = df.orderBy(F.col('datetime'))
from pyspark.sql import functions as F 
df = df.groupBy('ticker').agg(F.count("*").alias("num_of_bars")).count()

In [40]:
df = spark.read.parquet("s3a://lakehouse/bronze/ohlcv/1min")
count = df.groupBy('ticker').count()
count.count()

134

In [33]:
df = spark.read.parquet("s3a://lakehouse/bronze/ohlcv/1min")
df.show()

+-------------------+-----+-----+-----+-----+------+---------+------+----------+
|           datetime| Open| High|  Low|Close|Volume|   source|ticker|      date|
+-------------------+-----+-----+-----+-----+------+---------+------+----------+
|1041240600000000000|18.85|19.01|18.82| 19.0| 69211|pitrading|  AMZN|2026-06-04|
|1041240660000000000| 19.0|19.09|18.98|19.04| 53500|pitrading|  AMZN|2026-06-04|
|1041240720000000000|19.04|19.05|18.95|19.01| 61473|pitrading|  AMZN|2026-06-04|
|1041240780000000000|19.03|19.04|18.93|18.94| 80975|pitrading|  AMZN|2026-06-04|
|1041240840000000000|18.97|18.99|18.94|18.98| 28550|pitrading|  AMZN|2026-06-04|
|1041240900000000000|18.96|19.01|18.96|18.97| 67100|pitrading|  AMZN|2026-06-04|
|1041240960000000000|18.98|19.02|18.97| 19.0| 80512|pitrading|  AMZN|2026-06-04|
|1041241020000000000|19.03|19.05|18.99|19.02|140175|pitrading|  AMZN|2026-06-04|
|1041241080000000000|19.01|19.04|19.01|19.04| 22643|pitrading|  AMZN|2026-06-04|
|1041241140000000000|19.03|1